In [13]:
import pandas as pd
import glob
from datetime import timedelta

# ================= CONFIG =================
FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"
MIN_ANALYSTS = 5
TOP_N = 10
PERIOD_DAYS = 14     # 7 = 1W, 14 = 2W, 30 = 1M

# ================= LOAD DATA =================
files = sorted(glob.glob(f"{FOLDER}\\*_YahooConsensus.csv"))

df_all = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

df_all["Date"] = pd.to_datetime(df_all["Date"])

# Filter by analyst count
df_all = df_all[df_all["#Count"] >= MIN_ANALYSTS].copy()

# ================= DATE HELPERS =================
def get_asof_date(dates, target_date):
    eligible = dates[dates <= target_date]
    return eligible.max() if not eligible.empty else None

def period_label(days):
    if days % 30 == 0:
        return f"{days//30}M"
    elif days % 7 == 0:
        return f"{days//7}W"
    else:
        return f"{days}D"

P_LABEL = period_label(PERIOD_DAYS)

In [14]:
def tp_change_rank(df, tp_col, label):
    """
    tp_col: 'TP(Min)' / 'TP(Avg)' / 'TP(Max)'
    """

    latest_date = df["Date"].max()
    past_target = latest_date - timedelta(days=PERIOD_DAYS)
    past_date = get_asof_date(df["Date"], past_target)

    if past_date is None:
        print(f"\n{label}: Not enough history")
        return

    # One TP record per Ticker per Date (TP same across FYs)
    df_latest = (
        df[df["Date"] == latest_date]
        .drop_duplicates("YAHOO Ticker")
    )

    df_past = (
        df[df["Date"] == past_date]
        .drop_duplicates("YAHOO Ticker")
    )

    # MERGE ONLY ON TICKER (NO FIN YR)
    merged = df_latest.merge(
        df_past,
        on="YAHOO Ticker",
        suffixes=("_L", "_P")
    )

    # Valid TP filter
    merged = merged[
        (merged[f"{tp_col}_P"] > 0) &
        (merged[f"{tp_col}_L"] > 0)
    ]

    # Safe % change
    merged["TP_Change"] = (
        merged[f"{tp_col}_L"] / merged[f"{tp_col}_P"] - 1
    )

    merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)

    out_cols = [
        "YAHOO Ticker",
        "TP_Change",
        "#Count_L"
    ]

    # ================= TOP =================
    print(f"\n{label} – Highest")
    print(
        merged.sort_values("TP_Change", ascending=False)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

    # ================= BOTTOM =================
    print(f"\n{label} – Lowest")
    print(
        merged.sort_values("TP_Change", ascending=True)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

tp_change_rank(df_all, "TP(Min)", f"{P_LABEL} Min Target Price Revision")
tp_change_rank(df_all, "TP(Avg)", f"{P_LABEL} Avg Target Price Revision")
tp_change_rank(df_all, "TP(Max)", f"{P_LABEL} Max Target Price Revision")



2W Min Target Price Revision – Highest
YAHOO Ticker  TP_Change  #Count_L
         MCX   0.416667         7
  MANAPPURAM   0.368421         7
       AFFLE   0.118421         9
      ECLERX   0.084337        10
      NUVAMA   0.076087         8
  METROPOLIS   0.072508        18
    AXISBANK   0.072034        33
      CAMPUS   0.069767         7
  PHOENIXLTD   0.068966        19
  MUTHOOTFIN   0.068000        10

2W Min Target Price Revision – Lowest
YAHOO Ticker  TP_Change  #Count_L
    FIVESTAR  -0.325758         7
    GOCOLORS  -0.272727         6
     BRIGADE  -0.222222        13
    TATACHEM  -0.192308         7
   LICHSGFIN  -0.145833        24
   ITCHOTELS  -0.130435        10
    GREENPLY  -0.125000        13
   ACMESOLAR  -0.124242         6
  BLUESTARCO  -0.102337        21
       DIXON  -0.102146        27

2W Avg Target Price Revision – Highest
YAHOO Ticker  TP_Change  #Count_L
         MCX   0.185477         7
        GAIL   0.131068        20
  KARURVYSYA   0.125448        

C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1922088355.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1922088355.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1922088355.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed 

In [15]:
# ================= Rev MOMENTUM FUNCTION =================
def rev_change_rank(df, rev_col, label):
    """
    rev_col: 'Rev.(Min)' / 'Rev.(Avg)' / 'Rev.(Max)'
    """

    latest_date = df["Date"].max()
    past_target = latest_date - timedelta(days=PERIOD_DAYS)
    past_date = get_asof_date(df["Date"], past_target)

    if past_date is None:
        print(f"\n{label}: Not enough history")
        return

    df_latest = df[df["Date"] == latest_date]
    df_past = df[df["Date"] == past_date]

    # MERGE ON TICKER + FIN YEAR (CRITICAL)
    merged = df_latest.merge(
        df_past,
        on=["YAHOO Ticker", "FIN Yr"],
        suffixes=("_L", "_P")
    )

    merged = merged[
        (merged[f"{rev_col}_P"] > 0) &
        (merged[f"{rev_col}_L"] > 0)
    ]
    
    # Safe growth calculation
    merged["Rev_Change"] = (
        merged[f"{rev_col}_L"] / merged[f"{rev_col}_P"] - 1
    )

    # old = merged[f"{eps_col}_P"]
    # new = merged[f"{eps_col}_L"]
    # merged["EPS_Change"] = (new - old) / (old.abs() + 1e-6)
    # merged["EPS_Change"] = merged["EPS_Change"].clip(-5, 5)  # To clip outliers

    merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)

    out_cols = [
        "YAHOO Ticker",
        "FIN Yr",
        "Rev_Change",
        "#Count_L"
    ]

    # ================= TOP =================
    print(f"\n{label} – Highest")
    print(
        merged.sort_values("Rev_Change", ascending=False)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

    # ================= BOTTOM =================
    print(f"\n{label} – Lowest")
    print(
        merged.sort_values("Rev_Change", ascending=True)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

# ================= IDENTIFY Y1 & Y2 =================
fy_sorted = sorted(df_all["FIN Yr"].unique())
Y1 = fy_sorted[0]
Y2 = fy_sorted[1]

print("Y1:", Y1, "| Y2:", Y2)

# ================= RUN FOR FY26 (Y1) =================
df_y1 = df_all[df_all["FIN Yr"] == Y1]

rev_change_rank(df_y1, "Rev.(Min)", f"{P_LABEL} FY26 Rev. Min Revision")
rev_change_rank(df_y1, "Rev.(Avg)", f"{P_LABEL} FY26 Rev. Avg Revision")
rev_change_rank(df_y1, "Rev.(Max)", f"{P_LABEL} FY26 Rev. Max Revision")

# ================= RUN FOR FY27 (Y2) =================
df_y2 = df_all[df_all["FIN Yr"] == Y2]

rev_change_rank(df_y2, "Rev.(Min)", f"{P_LABEL} FY27 Rev. Min Revision")
rev_change_rank(df_y2, "Rev.(Avg)", f"{P_LABEL} FY27 Rev. Avg Revision")
rev_change_rank(df_y2, "Rev.(Max)", f"{P_LABEL} FY27 Rev. Max Revision")

C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\3351020935.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\3351020935.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\3351020935.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed 

Y1: FY26 | Y2: FY27

2W FY26 Rev. Min Revision – Highest
YAHOO Ticker FIN Yr  Rev_Change  #Count_L
         MCX   FY26    0.203202         7
         VBL   FY26    0.129719        23
     WELCORP   FY26    0.119453         5
       LODHA   FY26    0.098857        19
  CASTROLIND   FY26    0.096762         5
  STARHEALTH   FY26    0.090202        14
        HEXT   FY26    0.086187        12
     BRIGADE   FY26    0.084916        13
    TVSMOTOR   FY26    0.073244        33
         ACC   FY26    0.071103        12

2W FY26 Rev. Min Revision – Lowest
YAHOO Ticker FIN Yr  Rev_Change  #Count_L
      KAYNES   FY26   -0.088556        11
         MGL   FY26   -0.054973        25
    GOCOLORS   FY26   -0.053612         6
       DIXON   FY26   -0.028775        27
        CDSL   FY26   -0.027963        12
       CLEAN   FY26   -0.027615        11
        GAIL   FY26   -0.026905        20
     PVRINOX   FY26   -0.026259        14
  MANAPPURAM   FY26   -0.025232         7
      NAZARA   FY26   -0.

C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\3351020935.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)


In [16]:
# ================= EPS MOMENTUM FUNCTION =================
def eps_change_rank(df, eps_col, label):
    """
    eps_col: 'EPS(Min)' / 'EPS(Avg)' / 'EPS(Max)'
    """

    latest_date = df["Date"].max()
    past_target = latest_date - timedelta(days=PERIOD_DAYS)
    past_date = get_asof_date(df["Date"], past_target)

    if past_date is None:
        print(f"\n{label}: Not enough history")
        return

    df_latest = df[df["Date"] == latest_date]
    df_past = df[df["Date"] == past_date]

    # MERGE ON TICKER + FIN YEAR (CRITICAL)
    merged = df_latest.merge(
        df_past,
        on=["YAHOO Ticker", "FIN Yr"],
        suffixes=("_L", "_P")
    )

    merged = merged[
        (merged[f"{eps_col}_P"] > 0) &
        (merged[f"{eps_col}_L"] > 0)
    ]
    
    # Safe growth calculation
    merged["EPS_Change"] = (
        merged[f"{eps_col}_L"] / merged[f"{eps_col}_P"] - 1
    )

    # old = merged[f"{eps_col}_P"]
    # new = merged[f"{eps_col}_L"]
    # merged["EPS_Change"] = (new - old) / (old.abs() + 1e-6)
    # merged["EPS_Change"] = merged["EPS_Change"].clip(-5, 5)  # To clip outliers

    merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)

    out_cols = [
        "YAHOO Ticker",
        "FIN Yr",
        "EPS_Change",
        "#Count_L"
    ]

    # ================= TOP =================
    print(f"\n{label} – Highest")
    print(
        merged.sort_values("EPS_Change", ascending=False)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

    # ================= BOTTOM =================
    print(f"\n{label} – Lowest")
    print(
        merged.sort_values("EPS_Change", ascending=True)
              .head(TOP_N)[out_cols]
              .to_string(index=False)
    )

# ================= IDENTIFY Y1 & Y2 =================
fy_sorted = sorted(df_all["FIN Yr"].unique())
Y1 = fy_sorted[0]
Y2 = fy_sorted[1]

print("Y1:", Y1, "| Y2:", Y2)

# ================= RUN FOR FY26 (Y1) =================
df_y1 = df_all[df_all["FIN Yr"] == Y1]

eps_change_rank(df_y1, "EPS(Min)", f"{P_LABEL} FY26 EPS Min Revision")
eps_change_rank(df_y1, "EPS(Avg)", f"{P_LABEL} FY26 EPS Avg Revision")
eps_change_rank(df_y1, "EPS(Max)", f"{P_LABEL} FY26 EPS Max Revision")

# ================= RUN FOR FY27 (Y2) =================
df_y2 = df_all[df_all["FIN Yr"] == Y2]

eps_change_rank(df_y2, "EPS(Min)", f"{P_LABEL} FY27 EPS Min Revision")
eps_change_rank(df_y2, "EPS(Avg)", f"{P_LABEL} FY27 EPS Avg Revision")
eps_change_rank(df_y2, "EPS(Max)", f"{P_LABEL} FY27 EPS Max Revision")

Y1: FY26 | Y2: FY27


C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed 


2W FY26 EPS Min Revision – Highest
YAHOO Ticker FIN Yr  EPS_Change  #Count_L
      NUVAMA   FY26    3.500000         8
     PVRINOX   FY26    1.600000        14
    PRESTIGE   FY26    0.552941        20
       DIXON   FY26    0.425116        27
      UTIAMC   FY26    0.401765        10
   ACMESOLAR   FY26    0.366667         6
   POLICYBZR   FY26    0.340000        24
         MCX   FY26    0.245161         7
         DLF   FY26    0.192308        23
      CAMPUS   FY26    0.175000         7

2W FY26 EPS Min Revision – Lowest
YAHOO Ticker FIN Yr  EPS_Change  #Count_L
        TMPV   FY26   -0.973593        24
       NYKAA   FY26   -0.400000        22
      NAZARA   FY26   -0.366667         9
    TATACHEM   FY26   -0.320000         7
    GOCOLORS   FY26   -0.278571         6
  STARHEALTH   FY26   -0.271000        14
  EUREKAFORB   FY26   -0.224000         8
         MGL   FY26   -0.219048        25
  IDFCFIRSTB   FY26   -0.200000        15
  INDUSINDBK   FY26   -0.171429        23

2W F

C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.replace([float("inf"), -float("inf")], 0).fillna(0)
C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_3276\1148842259.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed 